# 🚀 PharMinds PRO Brain (Qwen2.5-14B Edition)

This is the **High-Performance** version of your AI brain. It runs the massive **Qwen2.5-14B-Instruct** model on the cloud, giving you professional-grade medical reasoning for free.

### 🛠️ Hardware Requirements:
- **GPU**: T4 (Free Tier) or Better.
- **VRAM**: ~10GB (Fits perfectly in Colab's 15GB).

### 📖 Quick Setup:
1. Ensure **Runtime Type** is set to **T4 GPU**.
2. Run all cells.
3. Paste your **Ngrok Token** when prompted.

In [ ]:
# 1. Install High-Efficiency Dependencies
!pip install -q -U transformers bitsandbytes accelerate fastapi uvicorn pyngrok nest-asyncio

In [ ]:
# 2. Load the 14B Pro Model (Quantized)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "Qwen/Qwen2.5-14B-Instruct"

print(f"🚀 Loading the PRO Brain: {model_id}...")
print("Note: This model is 2x larger than the previous one. Please wait for the download to complete (approx 10 minutes).")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

print("\n✨ PRO Brain is online and ready for complex medical reasoning!")

In [ ]:
# 3. Pro-Grade API Server
from fastapi import FastAPI, Request
import nest_asyncio
from pyngrok import ngrok
import uvicorn
import json

app = FastAPI(title="PharMinds PRO Brain")

@app.post("/v1/chat/completions")
async def chat_completion(request: Request):
    data = await request.json()
    messages = data.get("messages", [])
    
    # Standard Qwen-2.5 Prompt Format
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=1024,
        temperature=0.1,
        do_sample=True
    )
    
    generated_ids = [ 
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    
    return {
        "choices": [
            {
                "message": {
                    "role": "assistant",
                    "content": response_text
                }
            }
        ]
    }

# 4. Secure Connection
print("\n--- ENTER YOUR NGROK AUTHTOKEN BELOW ---")
token = input("Paste Token Here: ")
ngrok.set_auth_token(token)

public_url = ngrok.connect(8000)
print(f"\n🔗 PRO URL: {public_url.public_url}")
print("Update your PharMinds .env with: VITE_CUSTOM_AI_URL=" + public_url.public_url)

nest_asyncio.apply()
uvicorn.run(app, host="0.0.0.0", port=8000)